In [ ]:
import pandas as pd
import numpy as np
import networkx as nx
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from scipy.sparse import csr_matrix, diags
import random

path = './data/Menstrual_cramps.xlsx'
SEED = 42

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)
device = torch.device('cpu')

node_df = pd.read_excel(path, sheet_name='경혈단독_count')

node_df['경혈명'] = node_df['경혈명'].astype(str).str.strip().str.replace('Kl', 'KI', regex=False)

invalid_nodes = ['non-acupoint', 'ashi-point', 'BV', 'CV', 'FV']
all_nodes_list = sorted(list(set(node_df[~node_df['경혈명'].isin(invalid_nodes)]['경혈명'].tolist())))
N = len(all_nodes_list)
print(f"최종 분석 대상 노드 개수: {N}개 (5개 특수 노드 제외 완료)")

feature_raw = pd.read_excel(path, sheet_name='경혈의 feature (속성)_number')

feature_raw['경혈명'] = feature_raw['경혈명'].astype(str).str.strip().str.replace('Kl', 'KI', regex=False)

cols_to_convert = feature_raw.columns.drop(['경혈명'])
for col in cols_to_convert:
    feature_raw[col] = pd.to_numeric(feature_raw[col], errors='coerce')

feature_df = feature_raw.set_index('경혈명')
feature_df_numeric = feature_df.select_dtypes(include=[np.number])

# Reindexing & Zero-padding (현재 노드 리스트 순서에 맞춤)
df_X_final = feature_df_numeric.reindex(all_nodes_list).fillna(0)
print(f"최종 Feature Matrix 크기: {df_X_final.shape}")

# 인접 행렬(Adj) 생성 및 GCN 정규화
edge_df = pd.read_excel(path, sheet_name='경혈pair_count')

for col in ['Source', 'Target']:
    edge_df[col] = edge_df[col].astype(str).str.strip().str.replace('Kl', 'KI', regex=False)

edge_df = edge_df[edge_df['Source'].isin(all_nodes_list) & edge_df['Target'].isin(all_nodes_list)]

G_gcn = nx.Graph()
G_gcn.add_nodes_from(all_nodes_list)
for _, row in edge_df.iterrows():
    G_gcn.add_edge(row['Source'], row['Target'], weight=row['Weight'])

adj = nx.adjacency_matrix(G_gcn, nodelist=all_nodes_list)
A_dense = adj.toarray().astype(np.float32)
A_bin_np = (A_dense > 0).astype(np.float32)

A_tilde = adj + nx.adjacency_matrix(nx.Graph([(i, i) for i in range(N)]))
D = np.array(A_tilde.sum(axis=1)).flatten()
D_inv_sqrt = np.power(D, -0.5)
D_inv_sqrt[np.isinf(D_inv_sqrt)] = 0
D_inv_sqrt_mat = diags(D_inv_sqrt)

A_hat_np = D_inv_sqrt_mat.dot(A_tilde).dot(D_inv_sqrt_mat).toarray().astype(np.float32)


# GCN 모델 정의 및 학습 (GAE: Graph AutoEncoder 방식)

X = torch.tensor(df_X_final.values, dtype=torch.float32, device=device)
A_hat = torch.tensor(A_hat_np, dtype=torch.float32, device=device)
A_bin = torch.tensor(A_bin_np, dtype=torch.float32, device=device)

class GCN(nn.Module):
    def __init__(self, in_dim, hid, out, a_hat):
        super().__init__()
        self.register_buffer('a_hat', a_hat)
        self.fc1 = nn.Linear(in_dim, hid)
        self.fc2 = nn.Linear(hid, out)
        self.act = nn.ReLU()
    def forward(self, X):
        H = self.a_hat @ X
        H = self.act(self.fc1(H))
        Z = self.a_hat @ H
        Z = self.fc2(Z)
        return Z

def gae_loss(Z, A_bin):
    logits = Z @ Z.T
    mask = torch.triu(torch.ones_like(A_bin, dtype=torch.bool), diagonal=1)
    pos = A_bin[mask].sum()
    neg = mask.sum() - pos
    pos_weight = (neg / (pos + 1e-8)).clamp(max=1e6)
    return F.binary_cross_entropy_with_logits(logits[mask], A_bin[mask], pos_weight=pos_weight)

model = GCN(X.shape[1], 32, 8, A_hat).to(device)
opt = optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)

print("\n GCN 학습 시작")
for epoch in range(300):
    model.train()
    opt.zero_grad()
    Z = model(X)
    loss = gae_loss(Z, A_bin)
    loss.backward()
    opt.step()
    if (epoch+1) % 50 == 0:
        print(f"Epoch {epoch+1}/300 | Loss: {loss.item():.4f}")

model.eval()
with torch.no_grad():
    Z_final = model(X).cpu().numpy()

np.save('GCN_Z_embeddings.npy', Z_final)
print(f"\n 임베딩 저장됨: {Z_final.shape}")

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import normalize

Z = np.load('GCN_Z_embeddings.npy')

# (a) 유클리드
def eval_kmeans_euclid(Z, K):
    km = KMeans(n_clusters=K, n_init=20, max_iter=500, random_state=42)
    labels = km.fit_predict(Z)
    sil = silhouette_score(Z, labels, metric='euclidean')
    return sil, labels

# (b) 코사인: L2 정규화 후 평가
Z_norm = normalize(Z, norm='l2', axis=1)
def eval_kmeans_cosine(Zn, K):
    km = KMeans(n_clusters=K, n_init=20, max_iter=500, random_state=42)
    labels = km.fit_predict(Zn)
    sil = silhouette_score(Zn, labels, metric='cosine')
    return sil, labels

results = []
for K in [2,3,4,5,6,7,8,9]:
    sE, _ = eval_kmeans_euclid(Z, K)
    sC, _ = eval_kmeans_cosine(Z_norm, K)
    results.append((K, sE, sC))

print("K | Silhouette(Euclid) | Silhouette(Cosine)")
for K, sE, sC in results:
    print(f"{K:>1} | {sE: .4f}           | {sC: .4f}")

# 코사인 기준 최적 K 선택 예시
best = max(results, key=lambda x: x[2])
best_K = best[0]
_, best_labels = eval_kmeans_cosine(Z_norm, best_K)
np.save(f'GCN_kmeans_labels_K{best_K}.npy', best_labels)
print("선택 K:", best_K)


In [ ]:
import numpy as np
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score, adjusted_rand_score
from sklearn.preprocessing import normalize
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt

Z = np.load('GCN_Z_embeddings.npy')
Z_norm = normalize(Z, norm='l2', axis=1)

# 기존 선택
K = 5
km = KMeans(n_clusters=K, n_init=30, max_iter=500, random_state=42)
labels = km.fit_predict(Z_norm)

sil_cos = silhouette_score(Z_norm, labels, metric='cosine')
sil_euc = silhouette_score(Z, labels, metric='euclidean')
dbi = davies_bouldin_score(Z_norm, labels)
chi = calinski_harabasz_score(Z_norm, labels)
sizes = np.bincount(labels)
print(f"[K={K}] Silhouette(cos)={sil_cos:.4f} | Silhouette(euc)={sil_euc:.4f} | DBI={dbi:.3f} | CHI={chi:.1f}")
print("Cluster sizes:", sizes.tolist())

def kmeans_cos(Zn, K, seed):
    km = KMeans(n_clusters=K, n_init=30, max_iter=500, random_state=seed)
    return km.fit_predict(Zn)

labels_2 = kmeans_cos(Z_norm, K, 123)
labels_3 = kmeans_cos(Z_norm, K, 777)
print("ARI(seed42 vs 123):", adjusted_rand_score(labels, labels_2))
print("ARI(seed42 vs 777):", adjusted_rand_score(labels, labels_3))

tsne = TSNE(n_components=2, perplexity=15, learning_rate='auto', init='pca', random_state=42)
Z2 = tsne.fit_transform(Z_norm)
plt.figure(figsize=(6,5))
plt.scatter(Z2[:,0], Z2[:,1], c=labels, s=30, alpha=0.9)
plt.title(f"GCN Z (t-SNE), K={K}, Sil(cos)={sil_cos:.3f}")
plt.tight_layout(); plt.show()


In [ ]:
!pip install adjustText --quiet

import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.patches as mpatches
from sklearn.manifold import TSNE
import networkx as nx
from adjustText import adjust_text
from scipy.spatial.distance import pdist, squareform

tsne = TSNE(n_components=2, perplexity=15, learning_rate='auto', init='pca', random_state=42)
Z2_tsne = tsne.fit_transform(Z_norm)

dist_matrix = squareform(pdist(Z2_tsne))
eps = 0.15

partition = {name: labels[i] for i, name in enumerate(all_nodes_list)}
degree_dict = dict(G_gcn.degree())

cluster_info = {}
community_groups = {}

for k in range(best_K):
    members = [all_nodes_list[i] for i in np.where(labels == k)[0]]
    leader = max(members, key=lambda x: degree_dict.get(x, 0))
    cluster_info[k] = leader
    community_groups[k] = members

plt.figure(figsize=(20, 16))
cmap = mpl.colormaps['tab10'] if best_K <= 10 else mpl.colormaps['tab20']

pos_tsne = {}
np.random.seed(42)
for i, name in enumerate(all_nodes_list):
    neighbors = np.where(dist_matrix[i] < eps)[0]
    if len(neighbors) > 1:
        pos_tsne[name] = Z2_tsne[i] + np.random.normal(0, 0.05, size=2)
    else:
        pos_tsne[name] = Z2_tsne[i]

internal_edges = [(u, v) for u, v in G_gcn.edges() if partition[u] == partition[v]]
nx.draw_networkx_edges(G_gcn, pos_tsne, edgelist=internal_edges,
                       width=1.5, alpha=0.3, edge_color='#888888')

external_edges = [(u, v) for u, v in G_gcn.edges() if partition[u] != partition[v]]
nx.draw_networkx_edges(G_gcn, pos_tsne, edgelist=external_edges,
                       width=0.8, alpha=0.1, edge_color='#888888')

node_colors = [cmap(partition[name]) for name in all_nodes_list]
node_sizes = [1300 if name == cluster_info[partition[name]] else 500 for name in all_nodes_list]

nx.draw_networkx_nodes(G_gcn, pos=pos_tsne, node_color=node_colors,
                       node_size=node_sizes, alpha=0.85,
                       edgecolors='white', linewidths=1.2)

texts_to_adjust = []

for i, name in enumerate(all_nodes_list):
    x, y = pos_tsne[name]
    is_leader = (name == cluster_info[partition[name]])
    neighbors = np.where(dist_matrix[i] < eps)[0]

    if len(neighbors) > 1:
        # 중첩된 노드: adjust_text 처리 대상
        t = plt.text(x, y, name, fontsize=11 if is_leader else 9,
                     fontweight='bold' if is_leader else 'normal',
                     ha='center', va='center')
        texts_to_adjust.append(t)
    else:
        # 단독 노드: 노드 중앙에 고정
        plt.text(x, y, name, fontsize=11 if is_leader else 9,
                 fontweight='bold' if is_leader else 'normal',
                 ha='center', va='center', zorder=10)

if texts_to_adjust:
    adjust_text(texts_to_adjust,
                expand_points=(2.5, 2.5),
                expand_text=(2.0, 2.0),
                arrowprops=dict(arrowstyle='->', color='gray', lw=0.5, alpha=0.5))

legend_handles = [mpatches.Patch(color=cmap(i), label=f"Cluster {i+1} (Leader: {cluster_info[i]})") for i in range(best_K)]
plt.legend(handles=legend_handles, title="Cluster Info", loc='upper left', bbox_to_anchor=(1, 1), fontsize=12)

plt.title(f"GCN Embedding", fontsize=24, fontweight='bold', pad=30)
plt.axis('off')
plt.tight_layout()

plt.savefig("GCN_Final_Optimized_Edges.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
print(f"=== GCN 임베딩 군집 분석 상세 결과 (총 {K}개 군집) ===\n")

# K-means로 결정된 K개의 그룹 ID 순서대로 정렬하여 출력
for i in sorted(community_groups.keys()):
    leader_name = cluster_info[i]  # 위 시각화 코드에서 정의된 리더
    members = community_groups[i]  # 위 시각화 코드에서 정의된 멤버 리스트
    count = len(members)

    sorted_members = sorted(members)

    print(f" Cluster {i+1}")
    print(f"  - 대표 경혈(Leader): {leader_name}")
    print(f"  - 소속 경혈 수: {count}개")
    print(f"  - 경혈 리스트: {', '.join(sorted_members)}")
    print("-" * 50)

import pandas as pd

gcn_cluster_data = []
for i in sorted(community_groups.keys()):
    gcn_cluster_data.append({
        'Cluster': f"Cluster {i+1}",
        'Leader': cluster_info[i],
        'Count': len(community_groups[i]),
        'Members': ', '.join(sorted(community_groups[i]))
    })

df_gcn_summary = pd.DataFrame(gcn_cluster_data)
# df_gcn_summary.to_csv("GCN_Cluster_Summary.csv", encoding='utf-8-sig', index=False)